In [69]:
import json
import pandas as pd
import networkx as nx
from pathlib import Path

TRIPLES_FILE = Path("../data/results/community_triples.json")
GRAPH_FILE = Path("../data/results/knowledge_graph.graphml")
GRAPH_METRICS_FILE = Path("../data/results/knowledge_graph_metrics.json")
NODE_FILE = Path("../data/results/knowledge_graph_nodes.csv")
EDGE_FILE = Path("../data/results/knowledge_graph_edges.csv")



In [70]:
# Load the community triples
with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

print(f"Loaded {len(community_triples)} communities")

Loaded 16 communities


In [71]:
# Infer node type
def infer_node_type(node: str) -> str:
    node = str(node).lower().strip()

    if node == "network flow":
        return "flow"
    if node.isdigit():
        return "port"
    if "service" in node:
        return "service"
    if "flag" in node or node in {"syn", "ack", "rst", "fin", "psh"}:
        return "flag"
    if "duration" in node:
        return "duration"
    if "packet count" in node:
        return "packet_feature"
    if "traffic" in node or "connection" in node or "pattern" in node:
        return "behavior"
    return "entity"

In [72]:
# Build the graph
G = nx.DiGraph()
edge_counts = {}

for community_id, triples in community_triples.items():
    for t in triples:
        subject = str(t["subject"]).strip().lower()
        relation = str(t["relation"]).strip().lower()
        obj = str(t["object"]).strip().lower()

        if not subject or not relation or not obj:
            continue

        key = (subject, relation, obj)
        edge_counts[key] = edge_counts.get(key, 0) + 1

# Add nodes and edges
for (source, relation, target), count in edge_counts.items():
    G.add_node(source, node_type=infer_node_type(source))
    G.add_node(target, node_type=infer_node_type(target))

    G.add_edge(
        source,
        target,
        relation=relation,
        weight=count
    )

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 16
Edges: 15


In [73]:
# inspect top nodes by degree
degree_df = pd.DataFrame(
    [{"node": n, "degree": d, "node_type": G.nodes[n].get("node_type", "")} for n, d in G.degree()],
).sort_values("degree", ascending=False)

print("\nTop nodes by degree:")
print(degree_df.head(10).to_string(index=False))


Top nodes by degree:
                      node  degree node_type
              network flow      15      flow
           unknown service       1   service
     very short connection       1  behavior
            short duration       1  duration
    port scanning activity       1    entity
               ftp service       1   service
             long duration       1  duration
  observed traffic pattern       1  behavior
               dns service       1   service
brute force login activity       1    entity


In [74]:
# inspect top edges by weight
edges = []
for u, v, data in G.edges(data=True):
    edges.append({
        "source": u,
        "target": v,
        "relation": data.get("relation", ""),
        "weight": data.get("weight", 1)
    })

edges_df = pd.DataFrame(edges).sort_values("weight", ascending=False)

print("\nTop edges by weight:")
print(edges_df.head(20).to_string(index=False))



Top edges by weight:
      source                     target           relation  weight
network flow      very short connection     shows_behavior      12
network flow             short duration       has_duration      11
network flow brute force login activity indicates_activity       9
network flow     port scanning activity indicates_activity       7
network flow            unknown service    targets_service       6
network flow              long duration       has_duration       5
network flow   observed traffic pattern     shows_behavior       4
network flow                ftp service    targets_service       2
network flow               http service    targets_service       2
network flow                dns service    targets_service       1
network flow                rdp service    targets_service       1
network flow              https service    targets_service       1
network flow                ssh service    targets_service       1
network flow               smtp service 

In [75]:
# graph metrics
relation_counts = edges_df["relation"].value_counts().to_dict() if not edges_df.empty else {}

graph_metrics = {
    "n_nodes": int(G.number_of_nodes()),
    "n_edges": int(G.number_of_edges()),
    "avg_degree": float(sum(dict(G.degree()).values()) / G.number_of_nodes()) if G.number_of_nodes() > 0 else 0.0,
    "node_type_counts": degree_df["node_type"].value_counts().to_dict() if not degree_df.empty else {},
    "relation_counts": relation_counts
}

print("\nGraph metrics:")
print(graph_metrics)



Graph metrics:
{'n_nodes': 16, 'n_edges': 15, 'avg_degree': 1.875, 'node_type_counts': {'service': 8, 'behavior': 2, 'duration': 2, 'entity': 2, 'flow': 1, 'port': 1}, 'relation_counts': {'targets_service': 8, 'shows_behavior': 2, 'has_duration': 2, 'indicates_activity': 2, 'targets_port': 1}}


In [76]:
# Save graph and tables
nx.write_graphml(G, GRAPH_FILE)
degree_df.to_csv(NODE_FILE, index=False)
edges_df.to_csv(EDGE_FILE, index=False)

with open(GRAPH_METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(graph_metrics, f, indent=2)

print(f"\nSaved graph to {GRAPH_FILE}")
print(f"Saved node table to {NODE_FILE}")
print(f"Saved edge table to {EDGE_FILE}")
print(f"Saved graph metrics to {GRAPH_METRICS_FILE}")


Saved graph to ../data/results/knowledge_graph.graphml
Saved node table to ../data/results/knowledge_graph_nodes.csv
Saved edge table to ../data/results/knowledge_graph_edges.csv
Saved graph metrics to ../data/results/knowledge_graph_metrics.json
